# INSID3 medical benchmark 

## 0. Setup

In [ ]:
REPO_URL = "https://github.com/gary8564/insid3-medical-benchmark.git"
DRIVE_ROOT = "/content/drive/MyDrive/insid3-medical-benchmark"
REPO_DIR = "/content/insid3-medical-benchmark"
DATASET = "polyp"  # polyp | kidney_tumor | cardiac
MODEL_SIZE = "large"
IMAGE_SIZE = 768
SEED = 0
PREVIEW = None  # e.g. 8 for a first-pass; None = all 600

VIT_L = "dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth"
DOMAINS = ("polyp", "kidney_tumor", "cardiac")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path

drive_root = Path(DRIVE_ROOT)
for sub in ("processed", "pretrain", "results"):
    (drive_root / sub).mkdir(parents=True, exist_ok=True)
print("Drive layout:", drive_root)
print("ViT-L present:", (drive_root / "pretrain" / VIT_L).is_file())

In [ ]:
from pathlib import Path

repo = Path(REPO_DIR)
if not (repo / ".git").is_dir():
    !git clone --recurse-submodules {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
    !git -C {REPO_DIR} submodule update --init --recursive

assert (repo / "third_party" / "INSID3" / "models" / "insid3.py").is_file(), (
    f"INSID3 submodule missing at {repo / 'third_party' / 'INSID3'}; clone with --recurse-submodules"
)
%cd {REPO_DIR}

In [ ]:
# Colab already has torch, numpy, pillow, matplotlib, tqdm, scikit-learn.
# Do not `uv sync` or `uv pip install .[torch]`: pyproject pins CPython 3.10
# and would download a second torch on top of the runtime CUDA build.
%pip install -q uv
!uv pip install --system -q nibabel einops huggingface_hub pycocotools

<div style="background:#fff8e1;border-left:4px solid #f9a825;padding:0.8em 1em;margin:1em 0;color:#4a3c00;">
<strong>⚠️ DINOv3 access required</strong>. INSID3 uses a frozen DINOv3 encoder. Request a grant at <a href="https://github.com/facebookresearch/dinov3" style="color:#4a3c00;"><strong>facebookresearch/dinov3</strong></a>. Weights are gated and are not in this repo.
</div>

After approval, download **ViT-L** and place it in `pretrain/` at the repo root `pretrain/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth`

In [ ]:
from pathlib import Path

weights = Path(REPO_DIR) / "pretrain" / VIT_L
assert weights.is_file() and weights.stat().st_size > 0, (
    f"Missing {weights}. After DINOv3 access is granted, place {VIT_L} in pretrain/."
)
print("using", weights)

## 1. Download / preprocess benchmark datasets

Build the 2D cache used for in-context eval:

- **Kvasir-SEG** is a 2D colonoscopy set with paired polyp masks.
- **KiPA22** is a 3D abdominal CT challenge with kidney, vessels, and tumor labels.
- **ACDC** is a 3D cine cardiac MRI challenge with RV, myocardium, and LV cavity labels.

Kvasir-SEG is already 2D, so each endoscopy frame is kept with its binary polyp mask. KiPA22 and ACDC are volumes, so each case is preprocessed to one image–mask PNG. 

For KiPA, axial slice with the **most tumor** voxels (other structures become background) is chosen as the slice image per volume. 

For ACDC, we first read end-diastole from `Info.cfg` (largest LV fill), then the short-axis slice with the **largest LV cavity**.

<div style="background:#fff8e1;border-left:4px solid #f9a825;padding:0.8em 1em;margin:1em 0;color:#4a3c00;">
<strong>⚠️ Hugging Face login required</strong> for KiPA22 and ACDC.
</div>

In [ ]:
from huggingface_hub import login, notebook_login
import os

# Uses Colab secrets if you added HF_TOKEN; otherwise a login prompt.
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    notebook_login()

In [ ]:
from pathlib import Path

processed = Path(DRIVE_ROOT) / "processed"
missing = [
    name
    for name in DOMAINS
    if not any((processed / name / "images").glob("*.png"))
]
print("missing domains:", missing or "none")
if missing:
    !python src/data/prepare.py --processed-root {processed} --datasets {" ".join(missing)}

In [ ]:
from pathlib import Path

!rm -rf {REPO_DIR}/data/raw

processed = Path(DRIVE_ROOT) / "processed"
for name in DOMAINS:
    n = len(list((processed / name / "images").glob("*.png")))
    print(f"{name}: {n} images")
assert len(list((processed / "polyp" / "images").glob("*.png"))) >= 900
assert len(list((processed / "kidney_tumor" / "images").glob("*.png"))) >= 70
assert len(list((processed / "cardiac" / "images").glob("*.png"))) >= 100

## 2. Run in-context evaluation

Each domain is **600 random 1-shot episodes** (following INSID3’s lung/ISIC protocol): a random target and a different random reference, seed 0, and ViT-L at 768 px.

Set `DATASET` in Setup (`polyp`, `kidney_tumor`, or `cardiac`) and run the cells below. Each run writes Drive `results/<dataset>/metrics.json` (mIoU, Dice, per-episode scores) and predicted masks under `preds/`. A finished domain stays on Drive if the GPU disconnects.

To evaluate the next domain, change `DATASET` and run these cells again (polyp → kidney tumor → cardiac). `--preview N` runs a short subset to check the pipeline. 

In [ ]:
!python src/run_insid3.py --dataset {DATASET} \
  --input-dir {DRIVE_ROOT}/processed \
  --output-dir {DRIVE_ROOT}/results \
  --dry-run

In [ ]:
preview = f"--preview {PREVIEW}" if PREVIEW is not None else ""
!python src/run_insid3.py --dataset {DATASET} \
  --input-dir {DRIVE_ROOT}/processed \
  --output-dir {DRIVE_ROOT}/results \
  --model-size {MODEL_SIZE} --image-size {IMAGE_SIZE} --seed {SEED} --device cuda \
  {preview}

## 3. Inspect predictions

After a domain has `metrics.json`, the cells below overlay **reference + mask**, **target + ground truth**, and **target + prediction** — the same idea as the [INSID3 demo](https://colab.research.google.com/drive/1zCEqTS6lIbfaV3peNO5-m3U3FC8N2wNk?usp=sharing), with GT added for this benchmark.

The slider is ordered by IoU (0 = worst). Use it to look at failures; scores still come from `metrics.json`.

In [ ]:
%matplotlib inline

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from ipywidgets import IntSlider, interact

sys.path.insert(0, str(Path(REPO_DIR) / "third_party" / "INSID3"))
from utils.visualization import visualize_prediction_segmentation

processed = Path(DRIVE_ROOT) / "processed" / DATASET
metrics_path = Path(DRIVE_ROOT) / "results" / DATASET / "metrics.json"
pred_dir = Path(DRIVE_ROOT) / "results" / DATASET / "preds"


def _overlay(image: np.ndarray, mask: np.ndarray, color: tuple[float, float, float]) -> np.ndarray:
    out = image.astype(np.float32).copy()
    tint = np.array(color, dtype=np.float32) * 255.0
    out[mask] = 0.55 * out[mask] + 0.45 * tint
    return np.clip(out, 0, 255).astype(np.uint8)


def _load_metrics():
    if not metrics_path.is_file():
        raise FileNotFoundError(
            f"{metrics_path} missing — run the eval cell first (or --preview)."
        )
    payload = json.loads(metrics_path.read_text())
    items = sorted(payload["items"], key=lambda row: row["IoU"])
    return payload, items


def show_episode(rank: int = 0):
    """rank=0 is the worst IoU. Same 2-panel overlay as the official demo, plus GT."""
    payload, items = _load_metrics()
    row = items[int(rank)]
    ref_id, tgt_id = row["reference_id"], row["target_id"]
    pred_path = pred_dir / f"{int(row['episode_index']):04d}_{tgt_id}.png"

    ref_img = processed / "images" / f"{ref_id}.png"
    ref_mask = processed / "masks" / f"{ref_id}.png"
    tgt_img = processed / "images" / f"{tgt_id}.png"
    tgt_mask = processed / "masks" / f"{tgt_id}.png"

    print(
        f"{payload.get('dataset')} episode {row['episode_index']}: "
        f"ref={ref_id} → tgt={tgt_id}  IoU={row['IoU']:.3f}  Dice={row['Dice']:.3f}  "
        f"(n={int(payload['n'])}, mIoU={payload['mIoU']:.3f})"
    )

    visualize_prediction_segmentation(
        ref_img, ref_mask, tgt_img, pred_path, visualize=True
    )

    target_np = np.array(Image.open(tgt_img).convert("RGB"))
    gt = np.array(Image.open(tgt_mask).convert("L")) > 0
    pred = np.array(Image.open(pred_path).convert("L")) > 0
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    axes[0].imshow(target_np)
    axes[0].set_title("Target")
    axes[1].imshow(_overlay(target_np, gt, (0.2, 0.45, 0.95)))
    axes[1].set_title("Target + GT")
    axes[2].imshow(_overlay(target_np, pred, (0.15, 0.8, 0.35)))
    axes[2].set_title("Target + prediction")
    for ax in axes:
        ax.axis("off")
    plt.show()
    plt.close(fig)


payload, items = _load_metrics()
print(f"loaded {len(items)} episodes from {metrics_path}")
show_episode(0)

In [ ]:
# Drag to browse; 0 = worst IoU. 
_, items = _load_metrics()
interact(
    show_episode,
    rank=IntSlider(min=0, max=max(len(items) - 1, 0), step=1, value=0, description="worst→best"),
)